#Street Closures Transformation - Silver Layer

## Overview
This notebook implements the **Silver layer** transformation for Tel Aviv street closure data. It parses raw JSON payloads from the bronze layer, cleanses street names, extracts time ranges, normalizes the data structure, and produces a clean dataset ready for downstream analytics.

## Purpose
* Parse JSON payloads from bronze layer into structured columns
* Cleanse street names by removing symbols and special characters
* Extract and parse closure date ranges and time windows
* Handle complex time format parsing (dd/MM/yyyy H:mm with HH:MM-HH:MM ranges)
* Normalize data structure: explode from/to streets into separate rows
* Deduplicate and filter invalid street records
* Produce clean, conformed data for dimensional modeling

## Key Features
* **JSON Schema Parsing**: Predefined schema for consistent structure
* **Street Name Cleansing**: Removes symbols/special characters using regex whitelist
* **Complex Time Parsing**: Handles date + time range format (e.g., "15/03/2024 8:00 08:00-18:00")
* **Data Normalization**: Explodes from/to street pairs into individual rows (unpivoting)
* **Null Handling**: Graceful handling of missing timestamps and empty strings
* **Type Safety**: Uses `try_element_at()` for array access to prevent errors
* **Unicode Support**: Preserves Hebrew characters in street names
* **Event Lineage**: Maintains `event_id` from bronze layer for traceability

## Data Source
**Input**: `workspace.tel_aviv_municipality_raw.street_closures`
* Raw JSON payloads from CSV ingestion
* Contains closure metadata with from/to street pairs

## Output Table
**Target**: `workspace.tel_aviv_municipality_stg.street_closures`

**Schema**:
| Column | Type | Description |
|--------|------|-------------|
| `closure_id` | LONG | Unique identifier for the closure event |
| `street_id` | INT | Street identifier code |
| `street_name` | STRING | **Cleansed** street name (symbols removed) |
| `start_at` | TIMESTAMP | Closure start date and time |
| `end_at` | TIMESTAMP | Closure end date and time |
| `event_id` | STRING | Lineage: original event ID from bronze |

**Grain**: One row per street per closure event (normalized from from/to pairs)

## Data Transformation Logic

### Stage 1: JSON Parsing
**Input JSON Structure**:
```json
{
  "ID": "12345",
  "me_k_rechov": "101",        // From street ID
  "me_shem_rechov": "דיזנגוף",  // From street name
  "ad_k_rechov": "202",        // To street ID
  "ad_shem_rechov": "בן יהודה", // To street name
  "tr_from": "15/03/2024 8:00",
  "tr_to": "15/03/2024 18:00",
  "shaot": "08:00-18:00"       // Time range
}
```

### Stage 2: Street Name Cleansing
**Function**: `clean_all_signals(col_name)`
* **Removes**: All symbols and special characters
* **Keeps**: English letters (a-z, A-Z), numbers (0-9), Hebrew characters (א-ת), spaces
* **Pattern**: `[^a-zA-Z0-9א-ת\s]` (regex whitelist)
* **Null Handling**: Empty strings after cleaning are filtered out


### Stage 3: Timestamp Parsing
**Complex Format Handling**:
1. **Date Parsing**: `tr_from` and `tr_to` in format `dd/MM/yyyy H:mm`
2. **Time Range Extraction**: Regex captures `HH:MM-HH:MM` from `shaot` field
3. **Timestamp Construction**: Combines date + extracted time
4. **Default Handling**: If time is missing, defaults to `00:00`

**Example**:
```
tr_from: "15/03/2024 8:00"
tr_to:   "15/03/2024 18:00"
shaot:   "08:00-18:00"

Result:
start_at: 2024-03-15 08:00:00
end_at:   2024-03-15 18:00:00
```

**Edge Cases**:
* Missing `tr_to` → Uses `tr_from` date with end time
* Empty time range → Defaults to `00:00-00:00`
* Malformed `shaot` → Falls back to date-only timestamps

### Stage 4: Data Normalization (Unpivoting)
**Before** (1 row = 1 closure with from/to):
```
closure_id | from_id | from_name  | to_id | to_name
12345      | 101     | Dizengoff  | 202   | Ben Yehuda
```

**After** (2 rows = 1 per street):
```
closure_id | street_id | street_name
12345      | 101       | Dizengoff
12345      | 202       | Ben Yehuda
```

**Method**: Uses `F.explode()` with `F.array()` of structs to unpivot from/to into separate rows.

**Rationale**: Enables easier joins with business data on `street_name` in gold layer.

## Data Quality Filters
* `street_name IS NOT NULL` - Removes records with null street names
* `street_name != ''` - Removes records with empty street names after cleansing
* Implicit filtering via `try_element_at()` - Handles malformed arrays gracefully

## Processing Mode
**Current Mode**: Full overwrite
* Replaces entire silver table on each run
* Schema evolution enabled with `overwriteSchema = true`
* Suitable for datasets where full refresh is acceptable

## Design Principles
* **Schema-on-Write**: Parse JSON and enforce schema at silver layer
* **Data Quality**: Consistent null handling and symbol removal
* **Normalization**: Flatten nested structures for easier downstream joins
* **Unicode Support**: Handles Hebrew street names (א-ת character range)
* **Defensive Programming**: Uses `try_element_at()`, `coalesce()`, and graceful defaults
* **Lineage Preservation**: Maintains `event_id` from bronze layer

## Integration Notes
* **Upstream**: Depends on bronze layer (`workspace.tel_aviv_municipality_raw.street_closures`)
* **Downstream**: Feeds gold layer fact table (`fact_daily_business_compensation`)
* **Join Key**: `street_name` column used to join with business data
* **Date Expansion**: Gold layer further expands `start_at`/`end_at` into daily grain

## Known Data Patterns
* **Time Format Variations**: `shaot` field may contain ranges like "08:00-18:00" or empty strings
* **Hebrew Text**: Street names are primarily in Hebrew with some English
* **Closure Duration**: Typically same-day closures, but multi-day closures are supported
* **Street Pairs**: Each closure affects a range (from street → to street)

In [0]:
import json
import requests
import uuid
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType, DateType
from pyspark.sql import Row, functions as F
from delta.tables import DeltaTable

In [0]:
src_table = "workspace.tel_aviv_municipality_raw.street_closures"
destination_table = "workspace.tel_aviv_municipality_stg.street_closures"

json_schema = """
    ID STRING, 
    me_k_rechov STRING, 
    me_shem_rechov STRING, 
    ad_k_rechov STRING, 
    ad_shem_rechov STRING,
    tr_from STRING, 
    tr_to STRING, 
    shaot STRING
"""

In [0]:
def clean_all_signals(col_path):
    """
    Sanitizes string data by stripping all special characters and punctuation.
    
    This function uses a 'whitelist' approach via Regular Expression to ensure 
    only alphanumeric characters (English and Hebrew) and standard whitespace 
    remain. It is primarily used to resolve join mismatches caused by inconsistent 
    punctuation in street names across different municipality datasets.

    Logic:
    1. Trims leading and trailing whitespace.
    2. Removes any character that is NOT a letter (English/Hebrew), Number, or Space.
    
    Args:
        col_path (str): The column name or path to be sanitized.

    Returns:
        Column: A Spark SQL Column object containing only alphanumeric text and spaces.
    """
    raw_col = F.trim(F.col(col_path))
    # Regex: [^...] means "Not these characters". 
    clean_pattern = r"[^a-zA-Z0-9א-ת\s]"
    return F.regexp_replace(raw_col, clean_pattern, "")

In [0]:
raw_df = spark.table(src_table)

parsed_df = raw_df.withColumn("data", F.from_json(F.col("payload"), json_schema)) \
    .select(
        F.col("event_id"),
        F.col("data.ID").cast("long").alias("closure_id"),
        F.col("data.me_k_rechov").cast("int").alias("from_id"),
        clean_all_signals("data.me_shem_rechov").alias("from_name"),
        F.col("data.ad_k_rechov").cast("int").alias("to_id"),
        clean_all_signals("data.ad_shem_rechov").alias("to_name"),
        F.to_timestamp(F.col("data.tr_from"), "dd/MM/yyyy H:mm").cast("date").alias("_f_date"),
        F.to_timestamp(F.col("data.tr_to"), "dd/MM/yyyy H:mm").cast("date").alias("_t_date_raw"),
        F.col("data.shaot").alias("_shaot")
    ) \
    .withColumn("_t_date", F.coalesce(F.col("_t_date_raw"), F.col("_f_date")))

# TIMESTAMP BUILDING
hours_pattern = r"(\d{2}:\d{2}-\d{2}:\d{2})"

silver_processed = parsed_df.withColumn(
    "split_times", F.split(F.regexp_extract(F.col("_shaot"), hours_pattern, 1), "-")
).withColumn(
    "st_raw", F.expr("try_element_at(split_times, 1)")
).withColumn(
    "et_raw", F.expr("try_element_at(split_times, 2)")
).withColumn(
    "st_clean", F.when(F.trim(F.col("st_raw")) == "", F.lit(None)).otherwise(F.col("st_raw"))
).withColumn(
    "et_clean", F.when(F.trim(F.col("et_raw")) == "", F.lit(None)).otherwise(F.col("et_raw"))
).withColumn(
    "start_at",
    F.to_timestamp(
        F.concat(F.col("_f_date").cast("string"), F.lit(" "), F.coalesce(F.col("st_clean"), F.lit("00:00"))),
        "yyyy-MM-dd HH:mm"
    )
).withColumn(
    "end_at",
    F.to_timestamp(
        F.concat(F.col("_t_date").cast("string"), F.lit(" "), F.coalesce(F.col("et_clean"), F.lit("00:00"))),
        "yyyy-MM-dd HH:mm"
    )
)

# NORMALIZATION
silver_final = silver_processed.select(
    "closure_id", "start_at", "end_at", "event_id",
    F.explode(F.array(
        F.struct(F.col("from_id").alias("id"), F.col("from_name").alias("name")),
        F.struct(F.col("to_id").alias("id"), F.col("to_name").alias("name"))
    )).alias("street_info")
).select(
    "closure_id",
    F.col("street_info.id").alias("street_id"),
    F.col("street_info.name").alias("street_name"),
    "start_at",
    "end_at",
    "event_id"
).filter("street_name IS NOT NULL AND street_name != ''")

(
    silver_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(destination_table)
)